In [1]:
#数据导入
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

#数据导入部分
dir1 = "F:/IMU_computational_ethology/Data/2mouse_imu_83label/Data1_83"
dir2 = "F:/IMU_computational_ethology/Data/2mouse_imu_83label/Data2_e8"


def data_import(filedir):

    def extract_data(filedir):
        files = os.listdir(filedir)
        df_list = [pd.read_csv(os.path.join(filedir, f), header=None, names=list(range(23))) for f in files if f.endswith('.csv')]
        df_list = [df.drop(0) for df in df_list]
        df_list = [df.drop(df.columns[list(range(-1, -9, -1))], axis=1) for df in df_list]
        data = pd.concat(df_list, ignore_index=True)    
        return data

    def time_transform(arr):
        arr = np.array([t.replace(":", ".").split(".") for t in arr], dtype=float)
        secs = arr[:,0]*3600 + arr[:,1]*60 + arr[:,2] + arr[:,3]/1000
        return secs


    def frameloss_test(t):
        fl_count = 0
        lostframes = 0 
        for i in range(1, len(t)):
            if float(t[i]) - float(t[i-1]) >= 0.02:
                fl_count += 1
                lostframes += int((t[i] - t[i-1]) / 0.01) - 1
                #print(f"Frame loss between {i-1} and {i}: {t[i] - t[i-1]:.3f} seconds")        
        print(f"frame loss count: {fl_count}")
        print(f"total lost frames: {lostframes}")


    data = extract_data(filedir)
    time_onchip = time_transform([x.split(" ", 2)[2] for x in data[2]])
    time_sys = time_transform(data[0])
    frameloss_test(time_onchip)
    return np.concatenate((time_onchip.reshape(-1, 1), time_sys.reshape(-1, 1), data.iloc[:, 3:]), axis=1)

data1 = data_import(dir1)
data2 = data_import(dir2)

C:\Users\Shubai\AppData\Local\Temp\ipykernel_9280\3714764896.py:19: DtypeWarning: Columns (3,4,5,6,7,8,9,10,11,12,13,14,15,20,21) have mixed types. Specify dtype option on import or set low_memory=False.
  df_list = [pd.read_csv(os.path.join(filedir, f), header=None, names=list(range(23))) for f in files if f.endswith('.csv')]
C:\Users\Shubai\AppData\Local\Temp\ipykernel_9280\3714764896.py:19: DtypeWarning: Columns (3,4,5,6,7,8,9,10,11,12,13,14,15,20,21) have mixed types. Specify dtype option on import or set low_memory=False.
  df_list = [pd.read_csv(os.path.join(filedir, f), header=None, names=list(range(23))) for f in files if f.endswith('.csv')]
C:\Users\Shubai\AppData\Local\Temp\ipykernel_9280\3714764896.py:19: DtypeWarning: Columns (3,4,5,6,7,8,9,10,11,12,13,14,15,20,21) have mixed types. Specify dtype option on import or set low_memory=False.
  df_list = [pd.read_csv(os.path.join(filedir, f), header=None, names=list(range(23))) for f in files if f.endswith('.csv')]
C:\Users\Shub

frame loss count: 223
total lost frames: 93373806


C:\Users\Shubai\AppData\Local\Temp\ipykernel_9280\3714764896.py:19: DtypeWarning: Columns (3,4,5,6,7,8,9,10,11,12,13,14,15,20,21) have mixed types. Specify dtype option on import or set low_memory=False.
  df_list = [pd.read_csv(os.path.join(filedir, f), header=None, names=list(range(23))) for f in files if f.endswith('.csv')]
C:\Users\Shubai\AppData\Local\Temp\ipykernel_9280\3714764896.py:19: DtypeWarning: Columns (3,4,5,6,7,8,9,10,11,12,13,14,15,20,21) have mixed types. Specify dtype option on import or set low_memory=False.
  df_list = [pd.read_csv(os.path.join(filedir, f), header=None, names=list(range(23))) for f in files if f.endswith('.csv')]
C:\Users\Shubai\AppData\Local\Temp\ipykernel_9280\3714764896.py:19: DtypeWarning: Columns (3,4,5,6,7,8,9,10,11,12,13,14,15,20,21) have mixed types. Specify dtype option on import or set low_memory=False.
  df_list = [pd.read_csv(os.path.join(filedir, f), header=None, names=list(range(23))) for f in files if f.endswith('.csv')]
C:\Users\Shub

frame loss count: 335
total lost frames: 2699


In [1]:
#函数整合

import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from scipy.interpolate import PchipInterpolator
from scipy import signal
from scipy.signal import butter, filtfilt

def data_preprocess(data, t_diff= 6.56):

    def align_imu(data):
        """
        片上时间与系统时间对齐 + 掉帧插值
        data: ndarray, 列0=系统时间, 列2=片上时间, 列3:15=信号(12通道)
        返回: t_new, out, fs, drift
        """
        # 强制数值化，避免字符串输入
        t_sys = np.array(data[:, 1], dtype=float)
        t_chip = np.array(data[:, 0], dtype=float)


        sig = data[:, 2:8].astype(np.float64)
        
        # 1. 清洗片上时间异常
        dt = np.diff(t_chip)
        ok = np.concatenate(([True], (dt > 0) & (dt < 0.5)))
        t_sys, t_chip, sig = t_sys[ok], t_chip[ok], sig[ok]
        
        # 2. 晶振频偏拟合
        k, b = np.polyfit(t_sys, t_chip, 1)
        fs = 100.0 * k
        drift = (1.0 - k) * 100
        
        # 3. 反解平滑真实时间轴
        t_est = (t_chip - b) / k
        
        # 4. 等间隔目标轴
        dt = 1.0 / fs
        t_new = np.arange(t_est[0], t_est[-1] + dt/2, dt)
        
        out = np.zeros((len(t_new), sig.shape[1]))
        for i in range(sig.shape[1]):
            pchip = PchipInterpolator(t_est, sig[:, i])
            out[:, i] = pchip(t_new)
        
        return t_new, out, sig, fs, drift, t_est

    def resample(data, timestamps):
        """
        6轴IMU数据 99Hz → 256Hz 重采样
        输入: data (n,6), timestamps (n,)
        输出: data_resampled (m,6), t_resampled (m,)
        """
        data = np.asarray(data, dtype=float)
        timestamps = np.asarray(timestamps, dtype=float)
        
        duration = timestamps[-1] - timestamps[0]
        n_new = int(np.round(duration * 256.0)) + 1
        
        data_resampled = np.zeros((n_new, 6))
        for i in range(6):
            data_resampled[:, i] = signal.resample(data[:, i], n_new)
        
        t_resampled = np.linspace(timestamps[0], timestamps[-1], n_new)
        return data_resampled, t_resampled


    t_new, out, sig, fs, drift, t_est = align_imu(data)
    resampled_data, resampled_t = resample(sig, t_new)
    t_new_0 = resampled_t-resampled_t[0]

    ta = t_new_0 - t_diff 


    ta_p = ta[ta >= 0]
    ndrop = len(ta) - len(ta_p)
    imu_p= resampled_data[-len(ta_p):]
    
# =============================================================================
# 4D EKF-AHRS — 严格遵循 ahrs 库标准实现
# 状态: [q0, q1, q2, q3]  (传感器->地球/ENU, 标量在前)
# =============================================================================
    class EKF_AHRS:
        def __init__(self, fs=256.0, var_acc=0.002, var_gyr=0.75, frame='ENU'):
            self.fs = fs
            self.dt = 1.0 / fs
            self.var_acc = var_acc
            self.var_gyr = var_gyr
            self.frame = frame

            # 参考重力方向 (ENU: z向上; NED: z向下)
            self.g_ref = np.array([0.0, 0.0, 1.0]) if frame == 'ENU' else np.array([0.0, 0.0, -1.0])

            # 初始状态
            self.q = np.array([1.0, 0.0, 0.0, 0.0])
            self.P = np.eye(4) * 1.0

        @staticmethod
        def normalize(q):
            n = np.linalg.norm(q)
            return q / n if n > 1e-12 else q

        def _rotmat(self, q):
            """q: 传感器->地球. 返回 R (3x3), 满足 v_earth = R @ v_sensor"""
            q0, q1, q2, q3 = q
            return np.array([
                [1 - 2*(q2**2 + q3**2),   2*(q1*q2 - q0*q3),   2*(q1*q3 + q0*q2)],
                [2*(q1*q2 + q0*q3),   1 - 2*(q1**2 + q3**2),   2*(q2*q3 - q0*q1)],
                [2*(q1*q3 - q0*q2),   2*(q2*q3 + q0*q1),   1 - 2*(q1**2 + q2**2)]
            ])

        def _observation(self, q):
            """体坐标系中的预测重力方向: h = R.T @ g_ref"""
            R = self._rotmat(q)
            return R.T @ self.g_ref

        def _jacobian(self, q):
            """dh/dq, 3x4"""
            q0, q1, q2, q3 = q
            g = self.g_ref
            return np.array([
                [2*g[2]*(-q2), 2*g[2]*q3, 2*g[2]*(-q0), 2*g[2]*q1],
                [2*g[2]*q1,    2*g[2]*q0, 2*g[2]*q3,    2*g[2]*q2],
                [2*g[2]*0,     2*g[2]*(-2*q1), 2*g[2]*(-2*q2), 2*g[2]*0]
            ])

        def predict(self, w):
            """
            w: 去零偏后的传感器角速度 [wx, wy, wz] (rad/s)
            dq/dt = 0.5 * q ⊗ [0, w]  =>  q_new = (I + 0.5*Omega*dt) @ q
            """
            wx, wy, wz = w
            Omega = np.array([
                [ 0,   -wx, -wy, -wz],
                [ wx,   0,   wz, -wy],
                [ wy, -wz,   0,   wx],
                [ wz,  wy, -wx,   0 ]
            ])
            F = np.eye(4) + 0.5 * Omega * self.dt
            self.q = F @ self.q
            self.q = self.normalize(self.q)

            # 过程噪声 Q = (0.5*dt)^2 * Xi * Q_gyr * Xi.T
            q0, q1, q2, q3 = self.q
            Xi = np.array([
                [-q1, -q2, -q3],
                [ q0, -q3,  q2],
                [ q3,  q0, -q1],
                [-q2,  q1,  q0]
            ])
            Q = (0.5 * self.dt)**2 * Xi @ (np.eye(3) * self.var_gyr) @ Xi.T

            self.P = F @ self.P @ F.T + Q
            self.P = 0.5 * (self.P + self.P.T)

        def update(self, a_meas):
            """
            a_meas: 原始加速度计 [ax, ay, az] (g)
            """
            a_norm = np.linalg.norm(a_meas)
            if a_norm < 0.5:  # 自由落体/剧烈运动，跳过更新
                return

            # 归一化观测 (ahrs 库标准做法)
            z = a_meas / a_norm

            # 预测观测
            h = self._observation(self.q)

            # Jacobian
            H = self._jacobian(self.q)

            # 卡尔曼增益 (数值稳定)
            S = H @ self.P @ H.T + np.eye(3) * self.var_acc
            S = S + np.eye(3) * 1e-6

            try:
                K = np.linalg.solve(S, (self.P @ H.T).T).T
            except np.linalg.LinAlgError:
                K = self.P @ H.T @ np.linalg.pinv(S)

            # 状态更新: ahrs 库标准做法 —— 直接加法后归一化
            y = z - h
            self.q = self.q + K @ y
            self.q = self.normalize(self.q)

            # 协方差更新 (Joseph form)
            I_KH = np.eye(4) - K @ H
            self.P = I_KH @ self.P @ I_KH.T + K @ (np.eye(3) * self.var_acc) @ K.T
            self.P = 0.5 * (self.P + self.P.T)

            # 强制正定性
            try:
                eigvals, eigvecs = np.linalg.eigh(self.P)
                eigvals = np.clip(eigvals, 1e-12, None)
                self.P = eigvecs @ np.diag(eigvals) @ eigvecs.T
            except np.linalg.LinAlgError:
                pass

        def get_gravity_body(self):
            """体坐标系中的重力单位向量"""
            return self._observation(self.q)

        def get_euler(self):
            """ZYX 内旋欧拉角 (deg)"""
            q0, q1, q2, q3 = self.q
            roll = np.degrees(np.arctan2(2*(q0*q1 + q2*q3), 1 - 2*(q1**2 + q2**2)))
            sinp = 2*(q0*q2 - q3*q1)
            sinp = np.clip(sinp, -1.0, 1.0)
            pitch = np.degrees(np.arcsin(sinp))
            yaw = np.degrees(np.arctan2(2*(q0*q3 + q1*q2), 1 - 2*(q2**2 + q3**2)))
            return np.array([roll, pitch, yaw])

    def preprocess_segment(data, fs=256.0, apply_lpf=True, lpf_fc=30.0,
                        var_acc=0.002, var_gyr=0.75):
        a_raw = data[:, 0:3].astype(float)
        w_raw_deg = data[:, 3:6].astype(float)

        # 低通滤波
        if apply_lpf and fs > 100:
            nyq = fs / 2.0
            b, a = butter(4, lpf_fc / nyq, btype='low')
            a_raw = filtfilt(b, a, a_raw, axis=0)
            w_raw_deg = filtfilt(b, a, w_raw_deg, axis=0)

        w_raw = np.radians(w_raw_deg)

        # 陀螺仪零偏 (整段中位数)
        w_bias = np.median(w_raw, axis=0)
        w = w_raw - w_bias

        # EKF 姿态估计
        ekf = EKF_AHRS(fs=fs, var_acc=var_acc, var_gyr=var_gyr)

        n = len(data)
        aG = np.zeros((n, 3))
        r_deg = np.zeros((n, 3))

        for i in range(n):
            ekf.predict(w[i])
            ekf.update(a_raw[i])
            aG[i] = ekf.get_gravity_body()
            r_deg[i] = ekf.get_euler()

        # 加速度零偏
        acc_bias = np.median(a_raw - aG, axis=0)
        a = a_raw - acc_bias

        # 非重力加速度
        anG = a - aG

        # 输出 12 列: [aGx, aGy, aGz, anGx, anGy, anGz, wx, wy, wz, roll, pitch, yaw]
        out = np.concatenate([aG, anG, np.degrees(w), r_deg], axis=1)
        return out

    imu_preprocessed = preprocess_segment(imu_p, fs=256.0, apply_lpf=True, lpf_fc=15.0,
                                        var_acc=0.002, var_gyr=0.75)  
    
    return np.concatenate([ta_p.reshape(-1, 1), imu_preprocessed], axis=1)




preprocessed1 = data_preprocess(data1)
preprocessed2 = data_preprocess(data2)

NameError: name 'data1' is not defined

In [3]:
#将数据切割为60s的片段

def segmentation(imu):
    data_seg = []
    seg_len = 60
    edges = np.arange(imu[0, 0] // seg_len * seg_len, imu[-1, 0] + seg_len, seg_len)
    ix = np.searchsorted(imu[:, 0], edges)

    for s, e in zip(ix, ix[1:]):
        if e <= s: continue
        imui = imu[s:e]
        data_seg.append(imui)
    return data_seg


seg_data1 = segmentation(preprocessed1)
seg_data2 = segmentation(preprocessed2)

In [4]:
import numpy as np
import ruptures as rpt

def dissect_imu_single(data, penalty, sr=300.0, min_size=10):
    """
    单段 CPD，使用给定的全局 gamma 和 penalty。
    """
    n_samples = len(data)
    if n_samples < 2 * min_size:
        return [data]
    
    signal_cpd = np.concatenate([data[:, 1:4], data[:, 7:10]], axis=1)
    mu = np.mean(signal_cpd, axis=0)
    sigma = np.std(signal_cpd, axis=0) + 1e-12
    signal_z = (signal_cpd - mu) / sigma
    
    algo = rpt.KernelCPD(kernel="rbf", min_size=min_size)
    algo.fit(signal_z)
    bkps = algo.predict(pen=penalty)
    
    segments = []
    start = 0
    for end in bkps:
        if end - start >= min_size:
            segments.append(data[start:end, :])
        start = end
    
    return segments

In [5]:
segnum = 22
def CPD_segmentation(seg_data, segnum):
    CPDsegments = []
    for index in range(2, segnum):
        segs = dissect_imu_single(seg_data[index], penalty=5.6, sr=300, min_size=10)
        CPDsegments.extend(segs)
        print(f"Segment {index}: {len(segs)} sub-segs")
    all_lengths = [len(seg) / 100.0 * 1000.0 for seg in CPDsegments]

    print(f"全局中位段长: {np.median(all_lengths):.1f} ms")
    print(f"总片段数: {len(CPDsegments)}")
    return CPDsegments

CPD_segments1 = CPD_segmentation(seg_data1, segnum)
CPD_segments2 = CPD_segmentation(seg_data2, segnum)

Segment 2: 440 sub-segs
Segment 3: 453 sub-segs
Segment 4: 416 sub-segs
Segment 5: 429 sub-segs
Segment 6: 351 sub-segs
Segment 7: 380 sub-segs
Segment 8: 362 sub-segs
Segment 9: 400 sub-segs
Segment 10: 457 sub-segs
Segment 11: 448 sub-segs
Segment 12: 383 sub-segs
Segment 13: 382 sub-segs
Segment 14: 404 sub-segs
Segment 15: 396 sub-segs
Segment 16: 361 sub-segs
Segment 17: 355 sub-segs
Segment 18: 325 sub-segs
Segment 19: 394 sub-segs
Segment 20: 364 sub-segs
Segment 21: 344 sub-segs
全局中位段长: 320.0 ms
总片段数: 7844
Segment 2: 488 sub-segs
Segment 3: 528 sub-segs
Segment 4: 496 sub-segs
Segment 5: 504 sub-segs
Segment 6: 403 sub-segs
Segment 7: 408 sub-segs
Segment 8: 426 sub-segs
Segment 9: 317 sub-segs
Segment 10: 524 sub-segs
Segment 11: 501 sub-segs
Segment 12: 274 sub-segs
Segment 13: 309 sub-segs
Segment 14: 345 sub-segs
Segment 15: 418 sub-segs
Segment 16: 406 sub-segs
Segment 17: 350 sub-segs
Segment 18: 335 sub-segs
Segment 19: 378 sub-segs
Segment 20: 365 sub-segs
Segment 21: 3

In [7]:
import numpy as np
from numpy.fft import rfft, rfftfreq
from scipy.signal import find_peaks

def extract_features(data):
    """
    输入: n×13 numpy数组
        col 0  : 时间戳
        col 1-3: 重力加速度 (gx, gy, gz)      → 用于姿态估计、数据质量检查
        col 4-6: 非重力加速度 (ax, ay, az)     → 纯运动加速度，直接反映运动强度
        col 7-9: 角速度 (wx, wy, wz)
        col 10-12: 角度 (忽略)
    
    关键理解:
        - 重力加速度 g: 模长≈9.8，方向指向地心，用于估计设备姿态
        - 非重力加速度 a: 已去除重力，纯运动加速度，合加速度直接=运动强度
    输出: 23维特征向量
    """
    ts = data[:, 0]
    g = data[:, 1:4]      # 重力加速度 (姿态参考)
    a = data[:, 4:7]      # 非重力加速度 (纯运动)
    gyro = data[:, 7:10]  # 角速度
    
    n = len(data)
    dt = np.mean(np.diff(ts)) if n > 1 else 0.01
    fs = 1.0 / dt if dt > 0 else 100.0
    
    # ========== 派生信号 ==========
    # 运动强度: 非重力加速度的合加速度 (已去重力，直接反映运动)
    acc_mag = np.sqrt(np.sum(a ** 2, axis=1))
    # 旋转强度
    gyro_mag = np.sqrt(np.sum(gyro ** 2, axis=1))
    # 重力模长 (数据质量检查: 应稳定在~9.8)
    g_mag = np.sqrt(np.sum(g ** 2, axis=1))
    
    # ========== 姿态估计 (从重力加速度) ==========
    # 重力方向即"下"方向，由此推断俯仰角和横滚角
    # 归一化重力向量
    g_norm = g / (g_mag[:, None] + 1e-12)
    
    # 俯仰角 (pitch): 绕x轴，抬头为正
    pitch = data[:, 11]
    
    # 横滚角 (roll): 绕y轴，右倾为正  
    roll = data[:, 12]
    
    # ========== 频域分析 (基于运动加速度) ==========
    if n >= 8:
        fft_vals = np.abs(rfft(acc_mag - np.mean(acc_mag)))[1:]
        fft_freqs = rfftfreq(n, d=dt)[1:]
        psd = fft_vals ** 2
        psd_sum = np.sum(psd) + 1e-12
        psd_norm = psd / psd_sum
        
        dom_freq = fft_freqs[np.argmax(fft_vals)] if len(fft_vals) > 0 else 0.0
        spec_entropy = -np.sum(psd_norm * np.log2(psd_norm + 1e-12)) if len(psd_norm) > 0 else 0.0
    else:
        dom_freq = 0.0
        spec_entropy = 0.0
    
    # ========== 自相关 (周期性度量) ==========
    def autocorr(x, lag=1):
        xc = x - np.mean(x)
        c0 = np.sum(xc ** 2)
        return np.sum(xc[:-lag] * xc[lag:]) / c0 if len(x) > lag and c0 > 1e-12 else 0.0
    
    # ========== 峰值检测 ==========
    peaks, _ = find_peaks(acc_mag, distance=max(1, int(fs * 0.1))) if n >= 3 else ([], None)
    
    # ========== 跨轴相关 ==========
    corr_aa = np.corrcoef(a.T)
    mask = ~np.eye(3, dtype=bool)
    acc_corr = np.mean(np.abs(corr_aa[mask])) if np.all(np.isfinite(corr_aa)) else 0.0
    
    # ========== 运动方向特征 (非重力加速度各轴占比) ==========
    a_mean = np.mean(np.abs(a), axis=0)
    a_total = np.sum(a_mean) + 1e-12
    a_x_ratio = a_mean[0] / a_total
    a_y_ratio = a_mean[1] / a_total
    a_z_ratio = a_mean[2] / a_total
    
    # ========== 特征组装 ==========
    features = [
        # --- A. 运动剧烈程度 (基于非重力加速度) ---
        np.mean(acc_mag),           #  1. 合加速度均值
        np.std(acc_mag, ddof=1),    #  2. 合加速度标准差
        np.max(acc_mag),            #  3. 合加速度最大值
        np.sum(acc_mag ** 2),       #  4. 合加速度能量
        
        np.mean(gyro_mag),          #  5. 合角速度均值
        np.std(gyro_mag, ddof=1),   #  6. 合角速度标准差
        np.max(gyro_mag),           #  7. 合角速度最大值
        
        # --- B. 周期性/刻板行为 ---
        dom_freq,                   #  8. 主频率 (Hz)
        spec_entropy,               #  9. 频谱熵 (↓越周期)
        len(peaks),                 # 10. 峰值数
        autocorr(acc_mag, 1),       # 11. 加速度自相关(lag=1)
        autocorr(acc_mag, 2),       # 12. 加速度自相关(lag=2)
        
        # --- C. 姿态 (从重力加速度估计) ---
        np.mean(pitch),             # 13. 俯仰角均值
        np.std(pitch, ddof=1),      # 14. 俯仰角标准差
        np.mean(roll),              # 15. 横滚角均值
        np.std(roll, ddof=1),       # 16. 横滚角标准差
        np.mean(np.abs(gyro[:, 2])),# 17. 偏航角速度均值
        
        # --- D. 数据质量 (重力模长稳定性) ---
        np.mean(g_mag),             # 18. 重力模长均值 (应≈9.8)
        np.std(g_mag, ddof=1),      # 19. 重力模长标准差 (↓数据越稳定)
        
        # --- E. 运动方向分布 ---
        a_x_ratio,                  # 20. X轴运动占比
        a_y_ratio,                  # 21. Y轴运动占比
        a_z_ratio,                  # 22. Z轴运动占比
        
        # --- F. 协调性 ---
        acc_corr,                   # 23. 加速度轴间平均相关
    ]
    
    names = [
        "acc_mag_mean", 
        "acc_mag_std", 
        "acc_mag_max", 
        "acc_mag_energy",
        "gyro_mag_mean", 
        "gyro_mag_std", 
        "gyro_mag_max",
        "dominant_freq", 
        "spectral_entropy", 
        "peak_count",
        "acc_autocorr_lag1", 
        "acc_autocorr_lag2",
        "pitch_mean", 
        "pitch_std", 
        "roll_mean", 
        "roll_std", 
        "yaw_rate_mean",
        "gravity_mag_mean", 
        "gravity_mag_std",
        "a_x_ratio", 
        "a_y_ratio", 
        "a_z_ratio",
        "acc_inter_axis_corr",
    ]
    
    return np.array(features), names

In [8]:
features1 = [extract_features(seg)[0] for seg in CPD_segments1]
features2 = [extract_features(seg)[0] for seg in CPD_segments2]
names = extract_features(CPD_segments1[0])[1]

In [ ]:
# 保存中间变量到工作目录，供后续聚类模块读取
import sys
sys.path.insert(0, 'F:/WorkBuddy/IMU数据处理/src')
from data_io import save_mouse_data
import numpy as np

features1_arr = np.stack(features1, axis=0)
features2_arr = np.stack(features2, axis=0)

save_mouse_data('mouse1', features1_arr, CPD_segments1, output_dir='F:/WorkBuddy/IMU数据处理/outputs/intermediate_data')
save_mouse_data('mouse2', features2_arr, CPD_segments2, output_dir='F:/WorkBuddy/IMU数据处理/outputs/intermediate_data')
print('Saved intermediate data to F:/WorkBuddy/IMU数据处理/outputs/intermediate_data')
